In [ ]:
from google.colab import drive

# Mount Google Drive to save models, outputs, or intermediate files
drive.mount('/content/drive')

# Example folder in Google Drive to save results or CSV files
output_folder = '/content/drive/MyDrive/AML_Task1_Outputs/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Clone GitHub repository containing additional code/resources
# Replace the URL with your own repo if needed
!git clone https://github.com/Usernnaim/AML-task1-sentiment-analysis.git

fatal: destination path 'AML-task1-sentiment-analysis' already exists and is not an empty directory.


Import Libraries

In [ ]:
# Install necessary Python packages for NLP, machine learning, and data handling
# - nltk: natural language processing (tokenization, stopwords)
# - scikit-learn: ML models, TF-IDF, evaluation metrics
# - pandas: data manipulation
# - matplotlib / seaborn: visualization
!pip install --quiet nltk scikit-learn pandas matplotlib seaborn

# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Download required NLTK resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

Data loading

In [ ]:
# Download the data stored in a csv file
# If you're running all your experiments on a machine at home rather than using colab, then make sure you save it rather than repeatedly downloading it.

# training data
!wget "https://sussex.box.com/shared/static/xjewvhv527xq083j8ehtsjcijs1i4kra.csv" -O sentiment_analysis_training_data.csv
# validation data
!wget "https://sussex.box.com/shared/static/qv30jx3xfwer1va1k19nhndq2xaso67x.csv" -O sentiment_analysis_validation_data.csv
# test data (without labels)
!wget "https://sussex.box.com/shared/static/zu44986583mlocifzyx6dsmqe20g1hlh.csv" -O sentiment_analysis_test_data.csv

--2026-05-21 01:28:15--  https://sussex.box.com/shared/static/xjewvhv527xq083j8ehtsjcijs1i4kra.csv
Resolving sussex.box.com (sussex.box.com)... 74.112.186.157, 2620:117:bff0:12d::
Connecting to sussex.box.com (sussex.box.com)|74.112.186.157|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /public/static/xjewvhv527xq083j8ehtsjcijs1i4kra.csv [following]
--2026-05-21 01:28:16--  https://sussex.box.com/public/static/xjewvhv527xq083j8ehtsjcijs1i4kra.csv
Reusing existing connection to sussex.box.com:443.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://sussex.app.box.com/public/static/xjewvhv527xq083j8ehtsjcijs1i4kra.csv [following]
--2026-05-21 01:28:16--  https://sussex.app.box.com/public/static/xjewvhv527xq083j8ehtsjcijs1i4kra.csv
Resolving sussex.app.box.com (sussex.app.box.com)... 74.112.186.157, 2620:117:bff0:12d::
Connecting to sussex.app.box.com (sussex.app.box.com)|74.112.186.157|:443... connected.
HTTP request

Load datasets

In [ ]:
# Define paths to CSV files (change paths if needed)
train_path = 'sentiment_analysis_training_data.csv'
val_path = 'sentiment_analysis_validation_data.csv'
test_path = 'sentiment_analysis_test_data.csv'

# Load CSVs into pandas DataFrames
train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

# Extract text and labels as NumPy arrays
text_train = train_df['text'].values
labels_train = train_df['label'].values
text_val = val_df['text'].values
labels_val = val_df['label'].values
text_test = test_df['text'].values

# Print sample training data to inspect
print("Training set sample:")
print(train_df.sample(5))

Training set sample:
                                                    text  label
6311   it represents better-than-average movie-making...      1
3291   britney spears' phoniness is nothing compared ...      0
1969   Subject: order confirmation 11871528 - 01\r\nt...      0
10804  there seems to be no clear path as to where th...      0
2964   an intriguing look at the french film industry...      1


Random text visualisation

In [ ]:
def print_text(text, label):
  if label == 0:
    print (text, '\nlabel == 0')
  else:
    print (text, '\nlabel == 1')

import numpy as np
idx = np.random.randint(0, text_train.shape[0])
print_text(text_train[idx], labels_train[idx])

rice never clearly defines his characters or gives us a reason to care about them . 
label == 0


Calculating Confusion Matrix and exporting results

In [ ]:
from sklearn.metrics import confusion_matrix

def get_confusion_matrix(true_label, pred_label):
  """
  Calculate the confusion matrix for your predicted labels. See https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html
  :param pred_label: Array of predicted labels
  :param true_label: Array of corresponding ground truth (test) labels
  :return: Confusion matrix whose i-th row and j-th column entry indicates the number of samples with true label being i-th class and predicted label being j-th class.
  """
  return confusion_matrix(true_label, pred_label)

CSV save

In [ ]:
def save_as_csv(pred_labels, location = '.'):
    """
    Save the labels out as a .csv file
    :pred_labels: numpy array of shape (no_test_labels,) to be saved
    :param location: Directory to save results.csv in. Default to current working directory
    """
    assert pred_labels.shape[0]==1434, 'wrong number of labels, should be 1434 test labels'
    np.savetxt(location + '/results_task1.csv', pred_labels, delimiter=',')

Text Processing

In [ ]:

# Function to clean and preprocess text
def preprocess_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^a-z0-9\s]', '', text)  # Remove punctuation and special characters
    tokens = word_tokenize(text)  # Tokenize text into words
    tokens = [t for t in tokens if t not in stopwords.words('english')]  # Remove stopwords
    return ' '.join(tokens)  # Join tokens back into string

# Apply preprocessing to all datasets
train_df['clean_text'] = [preprocess_text(t) for t in text_train]
val_df['clean_text'] = [preprocess_text(t) for t in text_val]
test_df['clean_text'] = [preprocess_text(t) for t in text_test]

# Print some preprocessed examples
print("Sample preprocessed texts:")
print(train_df[['text', 'clean_text']].sample(3))

Sample preprocessed texts:
                                                    text  \
678    an emotionally strong and politically potent p...   
4488   Subject: $ 5 for cd ' s dvd ' s expires soon a...   
10075  Subject: re : pymt for 03 / 01 sales adjustmen...   

                                              clean_text  
678    emotionally strong politically potent piece ci...  
4488   subject 5 cd dvd expires soon half com free 5 ...  
10075  subject pymt 03 01 sales adjustment cowtrap ma...  


TF-IDF vectorisation

In [ ]:
# Convert cleaned text into numerical features using TF-IDF
tfidf = TfidfVectorizer(max_features=5000)  # Limit vocabulary size for efficiency
X_train = tfidf.fit_transform(train_df['clean_text'])
X_val = tfidf.transform(val_df['clean_text'])
X_test = tfidf.transform(test_df['clean_text'])

# Labels for training and validation
y_train = train_df['label']
y_val = val_df['label']

Train logic regression

In [ ]:
# Initialize and train a simple Logistic Regression classifier
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predict on validation set to check performance
y_val_pred = model.predict(X_val)

# Evaluate model using confusion matrix and classification report
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))
print("\nClassification Report:")
print(classification_report(y_val, y_val_pred))

Confusion Matrix:
[[478 215]
 [242 462]]

Classification Report:
              precision    recall  f1-score   support

           0       0.66      0.69      0.68       693
           1       0.68      0.66      0.67       704

    accuracy                           0.67      1397
   macro avg       0.67      0.67      0.67      1397
weighted avg       0.67      0.67      0.67      1397



Predict/save test predictions

In [ ]:
# Predict sentiment labels on the test set
test_pred = model.predict(X_test)

# Create output folder if it does not exist
import os
os.makedirs(output_folder, exist_ok=True)

# Save predictions in the required assignment CSV format
save_as_csv(test_pred, location=output_folder)

print("Predictions saved successfully.")

Predictions saved successfully.


feature importance analysis

In [ ]:
# Identify most important words for positive/negative sentiment
feature_names = tfidf.get_feature_names_out()
coefficients = model.coef_[0]

# Top 10 positive and negative words
top_pos = np.argsort(coefficients)[-10:]
top_neg = np.argsort(coefficients)[:10]

print("Top positive features:", [feature_names[i] for i in top_pos])
print("Top negative features:", [feature_names[i] for i in top_neg])

Top positive features: ['hilarious', 'solid', 'engrossing', 'beautiful', 'cinema', 'portrait', 'enjoyable', 'still', 'culture', 'performances']
Top negative features: ['bad', 'dull', 'boring', 'neither', 'fails', 'worst', 'feels', 'flat', 'script', 'jokes']
